# 02 — Preprocesamiento y Feature Engineering

Verifica el pipeline de normalización y genera features.


In [ ]:
import sys
from pathlib import Path

_bootstrap = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_bootstrap / 'src') not in sys.path:
    sys.path.insert(0, str(_bootstrap / 'src'))

from discurso_odio.paths import PROJECT_ROOT

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])

from discurso_odio.features.normalization import (
    normalize_for_model, has_accents_or_enye, has_uppercase,
    has_punctuation, has_mention_literal, has_url_literal, has_emoji_raw,
)
from discurso_odio.features.text_features import build_text_features



In [ ]:
corpus = load_split('corpus')
train = load_split('train')
print(len(corpus), 'registros')


## Verificación del pipeline de normalización


In [ ]:
orig = corpus['texto_original']
modelo = corpus['texto_modelo']
verification = pd.DataFrame({
    'propiedad': ['tildes/ñ', 'mayúsculas', 'puntuación', 'mención literal', 'URL literal', 'emoji crudo'],
    'texto_original': [
        orig.apply(has_accents_or_enye).sum(),
        orig.apply(has_uppercase).sum(),
        orig.apply(has_punctuation).sum(),
        orig.apply(has_mention_literal).sum(),
        orig.apply(has_url_literal).sum(),
        orig.apply(has_emoji_raw).sum(),
    ],
    'texto_modelo': [
        modelo.apply(has_accents_or_enye).sum(),
        modelo.apply(has_uppercase).sum(),
        modelo.apply(has_punctuation).sum(),
        modelo.apply(lambda t: '@usuario' in str(t)).sum(),
        modelo.apply(lambda t: 'http://url' in str(t)).sum(),
        modelo.apply(has_emoji_raw).sum(),
    ],
})
verification


## Feature engineering


In [ ]:
features_df = build_text_features(corpus)
feature_cols = [c for c in features_df.columns if c.startswith(('n_', 'upper_', 'exclamation', 'question', 'has_', 'es_', 'sarcasmo', 'ironia', 'contexto', 'dialecto'))]
features_df[feature_cols].describe().T


## Pesos de clase (solo train)


In [ ]:
weights = get_class_weights(train)
expected = {3: 1.3779, 2: 1.2099, 0: 0.9404, 1: 0.7224}
comparison = pd.DataFrame({
    'clase': [LABELS[i] for i in sorted(weights)],
    'peso_calculado': [round(weights[i], 4) for i in sorted(weights)],
    'peso_informe': [expected[i] for i in sorted(weights)],
})
comparison


In [ ]:
comparison.to_csv(REPORTS_METRICS / 'class_weights.csv', index=False)
print('Pesos exportados.')
